# Heatwave Hotspot Identification Using K-Means Clustering

**AI Use Case KJS-CES-01:** Climate Intelligence for Heatwave Monitoring, Prediction, and Early Warning  
**Collaborating Organization:** India Meteorological Department (IMD), Mumbai-Pune  
**Guide:** Dr. Radhika Kotecha, Professor and Head, Department of Information Technology, K J Somaiya Institute of Technology  

---

## 1. Problem Statement & Objectives
Heatwave occurrence and severity vary widely across regions of India. Looking at raw temperature maps makes it hard to see which areas behave similarly and which are persistent heat hotspots.

This project groups 1°×1° grid cells across India into clusters with similar heat behavior using **K-Means**, then identifies and ranks the clusters so that the hottest ones can be flagged as **heatwave hotspots**.

### Key Steps:
1. Load and clean IMD gridded daily maximum temperature (`.GRD` files).
2. Restrict analysis to the heatwave season (March to June).
3. Engineer per-grid-cell thermal behavioral features (excluding latitude and longitude).
4. Apply StandardScaler and choose optimal $k$ using Elbow method (inertia) and Silhouette score.
5. Label clusters by severity (**Low, Moderate, High, Extreme Hotspot**).
6. Validate clusters against known heatwave-prone zones of India (Vidarbha, Rajasthan, Odisha, Coastal AP).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in Python path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.load_data import load_heatwave_season_dataframe, get_available_raw_years
from src.features import compute_grid_features, CORE_HEAT_FEATURES
from src.cluster import fit_kmeans_and_label, evaluate_k_range, validate_against_known_heat_zones
from src.plots import plot_clustering_diagnostics, plot_elbow_and_silhouette, plot_cluster_radar, plot_hotspot_map, plot_region_breakdown

print("Modules imported successfully.")

## 2. Load and Clean IMD Gridded Temperature Data
We load multi-year IMD gridded daily maximum temperatures covering March to June for 2022 to 2024. IMD sentinel values (99.9) are converted to `NaN`, and ocean grid cells are filtered out.

In [ ]:
avail_years = get_available_raw_years(ROOT / "data/raw")
print(f"Available raw years: {avail_years}")

df_season = load_heatwave_season_dataframe(
    start_year=2022,
    end_year=2024,
    data_dir=ROOT / "data/raw",
    season_months=(3, 4, 5, 6)
)
print(f"Loaded {len(df_season):,} total records.")
print(f"Unique mainland Indian land grid cells: {df_season['grid_id'].nunique()}")
df_season.head()

## 3. Feature Engineering Per Grid Cell
Rather than clustering raw daily temperatures, we select four core heat behavioral features for K-Means clustering:
- `tmax_mean`: Seasonal mean maximum temperature (°C) — represents overall heat level
- `tmax_max`: Peak maximum temperature recorded (°C) — represents peak/extreme heat
- `hot_days_ge_40`: Number of days with Tmax >= 40°C — represents frequency of very high temperatures
- `heatwave_days`: Number of days satisfying IMD heatwave criteria — represents heatwave frequency

> **Feature Selection & Design Decisions:**
> 1. Additional metrics like `longest_hot_streak` and `tmax_std` are engineered and retained in the dataset for extended analysis, but are excluded from the K-Means input matrix to focus clustering on core thermal severity.
> 2. Latitude and longitude are **deliberately excluded** from the feature matrix so that clusters reflect thermal behavior rather than spatial adjacency.

In [ ]:
features_df = compute_grid_features(df_season, hot_threshold=40.0)
print(f"Engineered feature matrix shape: {features_df.shape}")
features_df[CORE_HEAT_FEATURES].describe().round(2)

## 4. Hyperparameter Selection: Evaluating k (k=2 through 8)
We evaluate K-Means for $k \in [2, 8]$ across four diagnostic criteria:
1. **Inertia (Elbow Curve)**: Within-cluster sum of squares (WCSS).
2. **Silhouette Score**: Cluster cohesion versus separation (higher is better).
3. **Davies-Bouldin Index**: Ratio of within-cluster distance to between-cluster separation (lower is better).
4. **Calinski-Harabasz Index**: Ratio of between-cluster to within-cluster dispersion (higher is better).

In [ ]:
diagnostics = evaluate_k_range(features_df, k_min=2, k_max=8)
print(diagnostics.to_string(index=False))

# 4-panel diagnostic evaluation (Inertia, Silhouette, Davies-Bouldin, Calinski-Harabasz)
fig_diag = plot_clustering_diagnostics(diagnostics, selected_k=4)
fig_diag.show()

### Justification for Choosing k = 4:
- **Elbow Inflection**: Substantial reduction in inertia from $k=2$ (522.27) to $k=4$ (206.97), with diminishing returns for $k > 4$ (drop to $k=5$ is only 42.14).
- **Silhouette Quality**: $k=4$ attains a strong score of **0.4580**, indicating robust cluster separation.
- **Davies-Bouldin Index**: Compact value of **0.7210** (lower is better), outperforming $k=3$ (0.7292) and $k=6$ (0.7879).
- **Calinski-Harabasz Index**: Attains a prominent local peak of **685.72**, maximizing between-cluster to within-cluster dispersion.
- **Operational Relevance**: Directly maps to IMD hazard tiers: **Low, Moderate, High, Extreme (Hotspot)**.

## 5. K-Means Clustering & Cluster Severity Labeling
We fit K-Means with $k=4$. Clusters are ranked monotonically by heat severity and assigned descriptive labels.

In [ ]:
labeled_df, profiles_df, meta = fit_kmeans_and_label(features_df, k=4)
print(f"Inertia / WCSS: {meta['inertia']:.2f}")
print(f"Silhouette Score: {meta['silhouette_score']:.4f}")
print(f"Davies-Bouldin Index: {meta['davies_bouldin']:.4f}")
print(f"Calinski-Harabasz Index: {meta['calinski_harabasz']:.2f}")
print(f"Primary Hotspot Label: {meta['hotspot_cluster_name']}")

profiles_df

## 6. Geospatial Hotspot Mapping
Visualizing the identified heatwave hotspots across India.

In [ ]:
fig_map = plot_hotspot_map(labeled_df, title="Identified Heatwave Hotspots in India (K-Means k=4)")
fig_map.show()

## 7. Cluster Thermal Footprint Comparison (Radar Chart)

In [ ]:
fig_radar = plot_cluster_radar(profiles_df)
fig_radar.show()

## 8. Validation Against Known IMD Heatwave Zones
We cross-reference the clusters with meteorological zones known for extreme summer temperatures.

In [ ]:
val_table = validate_against_known_heat_zones(labeled_df)
print(val_table.to_string())

fig_reg = plot_region_breakdown(labeled_df)
fig_reg.show()

## 9. Key Findings & Conclusion
1. **Accurate Hotspot Identification**: The top cluster (`Extreme (Hotspot)`) captures India's notorious heatwave corridors: Vidarbha (Nagpur, Chandrapur, Akola) and Western Rajasthan (Churu, Bikaner, Jodhpur), with an average of ~142 hot days (>=40°C), ~65 heatwave days, and max temperatures reaching 48.1°C.
2. **Natural Geographic Independence**: Although spatial coordinates were excluded, physical geography naturally emerged — Western Himalayas classified in `Low`, maritime coasts in `Moderate`, and arid continental interiors in `Extreme`.
3. **Decision-Support (Phase V)**: Emergency resource mobilization (cooling shelters, water tankers, hospital surge capacity) can be prioritized automatically for cells classified in the `Extreme` tier.